In [ ]:
%pip install --quiet opencv-python pillow pytesseract numpy pandas textblob


In [ ]:
import os, re, json, cv2, numpy as np
from PIL import Image
import pytesseract

# Keep any TensorFlow/transformers noise off (in case pulled by other libs)
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# If Tesseract isn't on PATH (Windows), uncomment and set the path:
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print("OpenCV:", cv2.__version__)


In [ ]:
def _tesseract_text(img_bgr) -> str:
    """Run Tesseract with robust defaults."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    thr  = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, 31, 11)
    return pytesseract.image_to_string(thr, config="--oem 3 --psm 6 -l eng")

def find_ingredient_section(pil_img: Image.Image) -> Image.Image:
    """Try to crop around the INGREDIENT(S) header; return full image if not found."""
    cv_image = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(cv_image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w * h < 500:
            continue
        roi = cv_image[y:y+h, x:x+w]
        text = pytesseract.image_to_string(roi, config="--oem 3 --psm 6 -l eng")
        if "INGREDIENT" in text.upper():  # matches INGREDIENT / INGREDIENTS
            extend = int(0.6 * h) + 300   # extend below to capture the list
            y2 = min(cv_image.shape[0], y + h + extend)
            x2 = min(cv_image.shape[1], x + w + 50)
            return pil_img.crop((x, y, x2, y2))
    return pil_img

def extract_ingredients(text: str) -> list[str]:
    """Extract a comma-separated ingredient list from OCR text."""
    text_norm = re.sub(r"\s+", " ", text).strip()
    m = re.search(r"ingredients?\s*[:\-]?\s*(.+)", text_norm, flags=re.IGNORECASE)
    if m:
        candidates = m.group(1)
    else:
        m2 = re.search(r"([A-Za-z0-9\s\(\)\[\]\-]+(?:,\s*[A-Za-z0-9\s\(\)\[\]\-]+)+)", text_norm)
        candidates = m2.group(1) if m2 else ""
    if not candidates:
        return []
    parts = [re.sub(r"^[\s\.\-:;]+|[\s\.\-:;]+$", "", x) for x in candidates.split(",")]
    return [p for p in parts if p and len(p) > 1]

def process_pil(pil_img: Image.Image) -> dict:
    """Full pipeline on a PIL image."""
    cropped = find_ingredient_section(pil_img)
    cv_img  = cv2.cvtColor(np.array(cropped), cv2.COLOR_RGB2BGR)
    raw     = _tesseract_text(cv_img)
    return {"ingredients": extract_ingredients(raw), "raw_text": raw.strip()}

def capture_image_from_camera(prefer_backend: str = "auto") -> Image.Image:
    """
    Capture from webcam and return a PIL image.
    prefer_backend: 'auto' | 'dshow' (Windows) | 'v4l2' (Linux)
    """
    if prefer_backend == "dshow":
        cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    elif prefer_backend == "v4l2":
        cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
    else:
        cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            cap.release()
            cap = cv2.VideoCapture(0, cv2.CAP_V4L2)

    if not cap.isOpened():
        raise RuntimeError("No camera found or cannot open camera. Use file input instead.")

    print("Press SPACE to capture, ESC to cancel...")
    while True:
        ok, frame = cap.read()
        if not ok:
            continue
        cv2.imshow("Capture Ingredient Image", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == 27:  # ESC
            cap.release(); cv2.destroyAllWindows()
            raise Exception("Image capture cancelled.")
        elif key == 32:  # SPACE
            cap.release(); cv2.destroyAllWindows()
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            return Image.fromarray(rgb)


In [ ]:
# --- Choose camera or file ---

USE_CAMERA = True          # set False to test with a file while debugging
PREFER_BACKEND = "dshow"   # "dshow" on Windows, "v4l2" on Linux, or "auto"

if USE_CAMERA:
    try:
        pil_img = capture_image_from_camera(prefer_backend=PREFER_BACKEND)
        result = process_pil(pil_img)
        print(json.dumps(result, ensure_ascii=False, indent=2))
    except Exception as e:
        print("Camera error:", e, "\nTip: set USE_CAMERA = False and provide a file path below.")
else:
    IMAGE_PATH = r"C:\path\to\your\photo.jpg"   # <-- set this
    pil_img = Image.open(IMAGE_PATH).convert("RGB")
    result = process_pil(pil_img)
    print(json.dumps(result, ensure_ascii=False, indent=2))
